# GeminiによるBLS文字起こし・18項目の自動評価

文字起こしには `gemini-3.5-transcribe`、意味判定には `gemini-3.8-flash` を使用します。
18項目を固定の等配点で採点し、参加者の発言に根拠がある項目だけ加点します。
「未検出」は実際の未実施を断定せず、「判定不能」も加点しません。API失敗時は点数を出しません。

プロジェクトルートで `uv run jupyter lab` を実行してください。
初期入力は保存済みTXTです。音声を選ぶと文字起こし後に採点します。
音声はGoogleのGemini APIへ送信されます。元のdata/は変更しません。
設定ごとのAPI応答を保存し、再実行ではキャッシュを利用します。

詳細・精度検証手順: [docs/bls_evaluation.md](docs/bls_evaluation.md)


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("プロジェクト内で起動してください")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# "audio"に変更すると音声から実行。既存TXTの生成設定は不明のため参考結果として扱う。
INPUT_MODE = "transcript"
TRANSCRIPT_PATH = PROJECT_ROOT / "outputs/transcription/gemini/1回目_右前_gemini.txt"
AUDIO_PATH = PROJECT_ROOT / "data/0604data/1回目_右前.wav"
if INPUT_MODE not in ("transcript", "audio"):
    raise ValueError("INPUT_MODEはtranscriptまたはaudio")
INPUT_PATH = TRANSCRIPT_PATH if INPUT_MODE == "transcript" else AUDIO_PATH
if not INPUT_PATH.is_file():
    raise FileNotFoundError(INPUT_PATH)
print("入力:", INPUT_PATH)


In [ ]:
import getpass
import os
from src.bls_pipeline import BLSPipeline, create_gemini_client

api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not api_key:
    api_key = getpass.getpass("Gemini API key（空欄なら既存キャッシュのみ）: ").strip()
client = create_gemini_client(api_key) if api_key else None
pipeline = BLSPipeline(
    client=client,
    vocabulary="revised",  # 比較時はbaseline。baselineは旧語彙の不可視文字を除去した設定。
    transcription_rpm=3, transcription_rpd=25,
    evaluation_rpm=3, evaluation_rpd=25,
)


In [ ]:
record = pipeline.evaluate_file(INPUT_PATH)
print(record["transcript"])
print("\n結果JSON:", record["result_path"])


In [ ]:
import pandas as pd
from IPython.display import display

evaluation = record["evaluation"]
status_labels = {"met": "達成", "not_detected": "未検出", "uncertain": "判定不能"}
print(f"自動評価: {evaluation['score']:.1f}点 / 達成 {evaluation['met_count']}/18 / 判定不能 {evaluation['uncertain_count']}項目")
display(pd.DataFrame([
    {
        "項目": item["id"], "内容": item["name"], "判定": status_labels[item["status"]],
        "根拠": " / ".join(e["quote"] for e in item["evidence"]),
        "理由": item["reason"], "検証上の留保": " / ".join(item["warnings"]),
    }
    for item in evaluation["items"]
]))


## 一括評価（初期状態では実行しない）

開発用は1〜3・6〜12回目、固定検証用は4・5・13〜20回目です。
語彙・評価基準を開発用で固定してから検証用を実行してください。
日上限で中断した場合、翌日以降の再実行で成功済みキャッシュを再利用します。


In [ ]:
RUN_BATCH = False
BATCH_SPLIT = "development"
if RUN_BATCH:
    if BATCH_SPLIT not in ("development", "validation"):
        raise ValueError("BATCH_SPLITが不正です")
    numbers = [1, 2, 3, *range(6, 13)] if BATCH_SPLIT == "development" else [4, 5, *range(13, 21)]
    batch_paths = [PROJECT_ROOT / "data/20260706" / f"20260706_{i}回目.wav" for i in numbers]
    batch_run = pipeline.run(batch_paths)
    print(batch_run["status"], batch_run["run_path"])
    if batch_run["errors"]:
        display(pd.DataFrame(batch_run["errors"]))


## 精度比較

`outputs/evaluation/bls/gold_20260706.json` に正解ラベルを保存します。
2〜5回目の達成ラベルはAGENTS.mdに基づきます。その他のnullをfalseに置き換えず、
音声確認後にtrue/falseを記入してください。人が確認した逐語録はreference_textに記入します。
これは研究用の正解作成であり、通常の自動採点に人の確認は必要ありません。


In [ ]:
from src.bls_benchmark import report
from src.bls_pipeline import read_json

REPORT_RUN = None  # pipeline.run()が出力したrun JSONのPathを指定
if REPORT_RUN is not None:
    gold = read_json(PROJECT_ROOT / "outputs/evaluation/bls/gold_20260706.json")
    comparison = report(gold, read_json(Path(REPORT_RUN)), split="validation")
    print("注釈済み比較項目:", comparison["evaluated_labeled_items"])
    print("点数MAE:", comparison["score_mae"], "判定不能率:", comparison["uncertain_rate"])
    display(pd.DataFrame(comparison["per_item"]))
    display(pd.DataFrame(comparison["coverage"]))
